In [1]:
import pandas as pd
import re
import math
import requests
import json
from io import StringIO
from variables import TOUR_NAME, POKEDATA_CSV, CATEGORY, LIMITLESS_LABS_BASE_ENDPOINT, LIMITLESS_LABS_TOUR_ID

In [2]:
response = requests.get(LIMITLESS_LABS_BASE_ENDPOINT.format(LIMITLESS_LABS_TOUR_ID, 'MA'))

In [3]:
deck_df = pd.DataFrame(response.json()['message'])

In [4]:
deck_df

,player_id,tp_id,name,country,drop_round,late,dqed,placement,points,wins,...,ties,opw,opw2,day2,topcut,dropped,decklist,deck_id,deck_name,icons
0,15775,2024,Dylan Kasturi,US,NaN,0,0,1.0,44,14,...,2,0.623724,0.606948,1,1,0,1,basic-box-m,Basic Box,ogerpon clefairy
1,855,791,Ben Dobberstein,US,NaN,0,0,2.0,42,14,...,0,0.670918,0.615191,1,1,0,1,ogerpon-meganium-hydrapple,Ogerpon Meganium Hydrapple,ogerpon hydrapple
2,22694,1629,Rohit Potti,US,NaN,0,0,3.0,38,12,...,2,0.617347,0.609747,1,1,0,1,dragapult-ex,Dragapult,dragapult
3,6257,444,Mark Dreitzler,US,NaN,0,0,4.0,38,12,...,2,0.585332,0.598748,1,1,0,1,slowking-scr,Slowking,slowking
4,6121,2795,Joshua Frink,US,NaN,0,0,5.0,38,12,...,2,0.651658,0.638577,1,1,0,1,alakazam-dudunsparce,Alakazam Dudunsparce,alakazam dudunsparce
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3117,32587,2271,Pavithra Ellison,US,3.0,0,0,3118.0,0,0,...,0,0.333333,0.490741,0,0,1,1,alakazam-dudunsparce,Alakazam Dudunsparce,alakazam dudunsparce
3118,55325,2349,Rachel Coombs,US,3.0,0,0,3119.0,0,0,...,0,0.333333,0.453704,0,0,1,1,lucario-hariyama,Lucario Hariyama,lucario-mega hariyama
3119,47841,1712,Max Norton,US,8.0,0,1,NaN,16,5,...,1,0.573438,0.582894,0,0,0,0,None,None,None
3120,55295,2250,Nicholas Crim Jr,US,2.0,0,1,NaN,1,0,...,1,0.750000,0.375000,0,0,1,0,None,None,None


In [5]:
archetype_dict = json.load(open("archetype.json"))

In [6]:
deck_df = pd.DataFrame({
    'Placement': deck_df['placement'],
    'Player': deck_df['name'],
    'Country': deck_df['country'],
    'Day 2': deck_df['day2'],
    'Deck': deck_df['deck_name'],
    'Variant': 'Standard'  # Set a constant value
})
deck_df = deck_df.dropna(subset=['Placement'])
deck_df['Placement'] = deck_df['Placement'].apply(lambda x: "Top {}".format(pow(2, math.ceil(math.log(x, 2)))))
deck_df['Deck'] = deck_df['Deck'].map(archetype_dict).fillna(deck_df['Deck'])

In [7]:
def clean_name(input_string):
    result = re.sub(r'\s*\[.*?\]\s*', '', input_string)
    result = re.sub(r'STATIC SEATING \(\d+\)\s*', '', result)
    result = re.sub(r'>.*?>', '', result)
    result = re.sub(r'>TABLE \d+ ', '', result)
    return result

In [8]:
pairings_df = pd.read_csv(StringIO(requests.get(POKEDATA_CSV).content.decode('utf-8')), sep='\t', header=None, encoding='utf-8')
pairings_df.rename(columns={0:'Player',1:'Opponent',2:'Result',3:'Points',4:'Round'}, inplace=True)
pairings_df['Player'] = pairings_df['Player'].apply(clean_name)
pairings_df['Opponent'] = pairings_df['Opponent'].apply(clean_name)
pairings_df = pairings_df[(pairings_df['Opponent'] != 'BYE') & (pairings_df['Opponent'] != 'LATE')]

In [9]:
# Check missing players
player_index = 0
for player in pairings_df['Player'].unique():
    if player not in deck_df['Player'].unique():
        print(player_index, player)
    player_index+=1

554 Max Norton
2727 Jake Lang
2865 Nicholas Crim Jr


In [10]:
# Step 1: Get list of players who appear more than once (case-insensitive)
players_lower = deck_df['Player'].str.lower()
players_more_than_once = players_lower.value_counts()
players_more_than_once = players_more_than_once[players_more_than_once > 1].index.tolist()

# Step 2: Process and rename
for i in deck_df.index:
    player = deck_df.at[i, 'Player']
    player_lower = player.lower()

    if player_lower in players_more_than_once:
        new_name = f"{player} {i + 1}"
        deck_df.at[i, 'Player'] = new_name

        first_round = True
        # Filter pairings_df by case-insensitive match
        matching_rows = pairings_df[pairings_df['Player'].str.lower() == player_lower]

        for index, row in matching_rows.iterrows():
            if row['Round'] == 1:
                if first_round:
                    first_round = False
                else:
                    break

            # Update the player's name in pairings
            pairings_df.loc[index, 'Player'] = new_name

            # Update their opponent’s row as well
            opponent_lower = row['Opponent'].lower()
            pairings_df.loc[
                (pairings_df['Player'].str.lower() == opponent_lower) & 
                (pairings_df['Round'] == row['Round']),
                'Opponent'
            ] = new_name


In [11]:
players_more_than_once

['andy zhang',
 'ryan harris',
 'eric chen',
 'eduardo melendez',
 'kevin kim',
 'xavier ramirez',
 'matthew hernandez',
 'william cooper',
 'tommy nguyen',
 'michael hunter',
 'hunter smith']

In [12]:
with pd.ExcelWriter(f'datasets/{TOUR_NAME}_{CATEGORY}.xlsx') as writer:
    pairings_df.to_excel(writer, sheet_name='pairings', index=False)
    deck_df.to_excel(writer, sheet_name='decks', index=False)
    # matchups_df.to_excel(writer, sheet_name='matchups', index=False)